# UNI2 embeddings on TENX99

In [ ]:
from pathlib import Path

import numpy as np
import torch
import tifffile
import zarr
from PIL import Image
import matplotlib.pyplot as plt

DATA_DIR = Path("/lustre/scratch126/cellgen/lotfollahi/so25/data/")
HEST_DIR = DATA_DIR / "hest_samples"
EMB_DIR = DATA_DIR / "embeddings"

SAMPLE_ID = "TENX99"

SRC_PIXEL_SIZE = 0.2125       # WSI level-0 um/px
DST_PIXEL_SIZE = 0.5          # target um/px
TARGET_PATCH_SIZE = 224       # px fed to UNI2-h
PATCH_SIZE_SRC = round(TARGET_PATCH_SIZE * (DST_PIXEL_SIZE / SRC_PIXEL_SIZE))  # 527 px @ level 0

ENCODER_NAME = "uni_v2"       # UNI2-h in the HEST/TRIDENT registry
BATCH_SIZE = 128

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

### Cells and patch centers

We take the canonical cell order and each cell's H&E-pixel centroid straight
from `TENX99_terra.npz`, so the UNI2-h rows are aligned
1:1 with the TERRA rows and no centroid recomputation is needed.

In [ ]:
terra = np.load(EMB_DIR / f"{SAMPLE_ID}_terra.npz")
cell_ids = terra["cell_ids"]

he_x = terra["he_x"].astype(np.float64)
he_y = terra["he_y"].astype(np.float64)
print(f"Loaded he_x/he_y for {len(cell_ids):,} cells from terra.npz (aligned to embedding order).")

centers = np.stack([he_x, he_y], axis=1)  # (N, 2) level-0 pixel (x, y)
assert len(centers) == len(cell_ids) and not np.isnan(centers).any()
print("centers shape:", centers.shape)

### Extract patches

For each cell we cut a `PATCH_SIZE_SRC` (527 px) window at WSI level 0 centered
on its `(he_x, he_y)` pixel centroid, then downscale to 224 px. Windows that run
off the slide edge are white-padded so every cell keeps a patch and the row
order stays aligned with TERRA. The 224 px `uint8` patches are held in memory
(~1.5 GB for 10k) and handed to UNI2-h's own eval transform next.

In [ ]:
wsi = zarr.open(
    tifffile.imread(HEST_DIR / "wsis" / f"{SAMPLE_ID}.tif", aszarr=True, level=0),
    mode="r",
)
H, W = wsi.shape[:2]
print("WSI level-0 shape:", wsi.shape)

half = PATCH_SIZE_SRC // 2


def crop_patch(cx, cy):
    """527 px level-0 window centered on (cx, cy), white-padded at slide edges,
    downscaled to 224 px (bilinear). Returns a (224, 224, 3) uint8 array."""
    x0, y0 = int(round(cx)) - half, int(round(cy)) - half
    x1, y1 = x0 + PATCH_SIZE_SRC, y0 + PATCH_SIZE_SRC
    vx0, vy0, vx1, vy1 = max(x0, 0), max(y0, 0), min(x1, W), min(y1, H)
    canvas = np.full((PATCH_SIZE_SRC, PATCH_SIZE_SRC, 3), 255, dtype=np.uint8)
    if vx1 > vx0 and vy1 > vy0:
        canvas[vy0 - y0 : vy1 - y0, vx0 - x0 : vx1 - x0] = wsi[vy0:vy1, vx0:vx1, :3]
    img = Image.fromarray(canvas).resize(
        (TARGET_PATCH_SIZE, TARGET_PATCH_SIZE), Image.BILINEAR
    )
    return np.asarray(img)


n_padded = 0
patches = np.empty((len(centers), TARGET_PATCH_SIZE, TARGET_PATCH_SIZE, 3), dtype=np.uint8)
for i, (cx, cy) in enumerate(centers):
    x0, y0 = int(round(cx)) - half, int(round(cy)) - half
    if x0 < 0 or y0 < 0 or x0 + PATCH_SIZE_SRC > W or y0 + PATCH_SIZE_SRC > H:
        n_padded += 1
    patches[i] = crop_patch(cx, cy)
    if (i + 1) % 2000 == 0:
        print(f"  {i + 1:,}/{len(centers):,} patches")

print(f"Done. {len(patches):,} patches; {n_padded} needed edge-padding.")

### QC: a few patches on their cells

Sanity check that patches are tissue (not background) and centered.

In [ ]:
rng = np.random.default_rng(0)
show = rng.choice(len(patches), size=8, replace=False)
fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, idx in zip(axes.ravel(), show):
    ax.imshow(patches[idx])
    ax.plot(TARGET_PATCH_SIZE / 2, TARGET_PATCH_SIZE / 2, "+", c="red", ms=10, mew=1.5)
    ax.set_title(f"cell {cell_ids[idx]}", fontsize=8)
    ax.axis("off")
fig.suptitle("UNI2-h input patches (112 um FOV -> 224 px), + marks the cell centroid")
fig.tight_layout()

### Load UNI2-h and embed

`encoder_factory("uni_v2")` downloads/loads `MahmoodLab/UNI2-h`. We run inference in batches under autocast and collect the 1536-dim CLS features.

In [ ]:
from trident.patch_encoder_models import encoder_factory

encoder = encoder_factory(ENCODER_NAME).to(DEVICE).eval()
eval_transform = encoder.eval_transforms
precision = encoder.precision
print("encoder:", encoder.enc_name, "| precision:", precision)

In [ ]:
@torch.inference_mode()
def embed_patches(patches_uint8):
    embs = []
    for i in range(0, len(patches_uint8), BATCH_SIZE):
        batch = patches_uint8[i : i + BATCH_SIZE]
        x = torch.stack(
            [eval_transform(Image.fromarray(p)) for p in batch]
        ).to(DEVICE)
        with torch.autocast(device_type="cuda", dtype=precision, enabled=(DEVICE == "cuda")):
            out = encoder(x)
        embs.append(out.float().cpu().numpy())
        if (i // BATCH_SIZE + 1) % 10 == 0:
            print(f"  {min(i + BATCH_SIZE, len(patches_uint8)):,}/{len(patches_uint8):,}")
    return np.concatenate(embs, axis=0)


uni_emb = embed_patches(patches)
print("uni_emb shape:", uni_emb.shape)
assert uni_emb.shape[0] == len(cell_ids)

### Save embeddings

Saved as `TENX99_uni2.npz` with `cell_ids` in the **same row order** as
`TENX99_terra.npz`, so the fusion can concatenate the two
embedding matrices directly.

In [ ]:
np.savez(
    EMB_DIR / f"{SAMPLE_ID}_uni2.npz",
    cell_ids=cell_ids,
    uni_emb=uni_emb,
)
# The UNI2-h rows must line up 1:1 with the TERRA rows for the concat probe.
assert np.array_equal(cell_ids, terra["cell_ids"])
print(f"Saved -> {EMB_DIR / f'{SAMPLE_ID}_uni2.npz'}")
print("uni_emb shape:", uni_emb.shape)

### Are the embeddings informative?

A near-1 average cosine similarity would mean the embeddings collapsed. We then
UMAP the UNI2-h embeddings and color by Leiden clusters to check for visible
morphological structure.

In [ ]:
Xn = uni_emb / np.linalg.norm(uni_emb, axis=1, keepdims=True)
n = Xn.shape[0]
S = Xn.sum(axis=0)
avg_cos = (S @ S - n) / (n * (n - 1))
print(f"uni_emb: average cosine similarity = {avg_cos:.3f}")

In [ ]:
import anndata as ad
import scanpy as sc

adata_uni = ad.AnnData(
    X=uni_emb.astype(np.float32),
    obs={"cell_id": [str(c) for c in cell_ids]},
)
adata_uni.obsm["X_uni2"] = uni_emb.astype(np.float32)
sc.pp.neighbors(adata_uni, use_rep="X_uni2")
sc.tl.leiden(adata_uni, resolution=0.6, flavor="igraph", n_iterations=2, directed=False)
sc.tl.umap(adata_uni)
sc.pl.umap(adata_uni, color="leiden", palette="tab20", title="UNI2-h embeddings")